Goal: clean the raw Amazon sales data and prepare it for analysis<br>
Input: the raw CSV in data<br>
Output: a cleaned dataset (saved in data)<br>

In [44]:
#importing required libraries
import pandas as pd

In [45]:
#load dataset
raw_data = pd.read_csv("../data/Amazon Sale Report.csv")
df = raw_data.copy()

In [46]:
#shape of data before cleaning
print(raw_data.shape)
print(df.shape)

(128976, 21)
(128976, 21)


In [47]:
#removing duplicates
print("Before removing duplicates",df.shape)
df = df.drop_duplicates()
print("After removing duplicates",df.shape)

Before removing duplicates (128976, 21)
After removing duplicates (128808, 21)


In [48]:
df.duplicated().value_counts()

False    128808
Name: count, dtype: int64

In [49]:
df.columns

Index(['index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel',
       'ship-service-level', 'Category', 'Size', 'Courier Status', 'Qty',
       'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code',
       'ship-country', 'B2B', 'fulfilled-by', 'New', 'PendingS'],
      dtype='object')

<h3>Removing useless columns</h3>
-New:       It is empty<br>
-PandingS:  It is empty<br>
-index:     Same as row numbers<br>
-currency:  Only one value INR

In [50]:
df = df.drop(columns=['index','currency','New', 'PendingS'])

In [51]:
df.shape

(128808, 17)

Handling Null 

In [52]:
#fulfilled-by column
df['fulfilled-by'].unique()

array(['Easy Ship', nan], dtype=object)

In [53]:
df['fulfilled-by'] = df['fulfilled-by'].fillna('Not Easy Ship')

In [54]:
df['fulfilled-by'].unique()

array(['Easy Ship', 'Not Easy Ship'], dtype=object)

In [55]:
# converting and fixing type of Date column
print(df['Date'].str.len().value_counts())
print(df.loc[df['Date'].str.len() != 8, 'Date'].head(10))
d1 = pd.to_datetime(df['Date'], format='%m-%d-%y', errors='coerce')
d2 = pd.to_datetime(df['Date'], format='%m-%d-%Y', errors='coerce')
df['Date'] = d1.fillna(d2)

print('NUll',df['Date'].isna().sum())           
print("Min and Max",df['Date'].min(), df['Date'].max()) 
# df['Date'] = pd.to_datetime(df['Date'], format='%m-%d-%y')  #, format='%m-%d-%y'
print(df['Date'].info())
min_date = df['Date'].min()
max_date = df['Date'].max()
print(f"\nDate range is from {min_date} to {max_date}")

Date
8     73699
10    55109
Name: count, dtype: int64
29926    04-12-2022
29927    04-12-2022
29928    04-12-2022
29929    04-12-2022
29930    04-12-2022
29931    04-12-2022
29932    04-12-2022
29933    04-12-2022
29934    04-12-2022
29935    04-12-2022
Name: Date, dtype: object
NUll 0
Min and Max 2022-03-31 00:00:00 2022-06-29 00:00:00
<class 'pandas.core.series.Series'>
Index: 128808 entries, 0 to 128975
Series name: Date
Non-Null Count   Dtype         
--------------   -----         
128808 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 2.0 MB
None

Date range is from 2022-03-31 00:00:00 to 2022-06-29 00:00:00


In [56]:
# adding saperate columns of Year, Month, and Year-month
df['Year'] = df['Date'].dt.year
df['Month'] =df['Date'].dt.month
df['Month Name']=df['Date'].dt.month_name()
df['Year-Month'] = df['Date'].dt.to_period('M')

Handling  outlier values<br>
Qty = 0 <br>
Amount = 0

In [57]:
#Qty = 0, Amount is Null and  cancelled by Status  
qty0 = df[(df['Qty'] == 0)&(df['Amount'].isna()) &(df['Status']=="Cancelled")]
qty0.shape
#Cancelled orders nothing was shipped

(7559, 21)

7559 rows have Qty =0 , Amount is null and order is cancelled


In [58]:
#Qty = 0,Amount is Null and not cancelled by Status
qty1 = df[(df['Qty'] == 0)&(df['Amount'].isna()) &(df['Status']!="Cancelled")]
qty1.shape

(106, 21)

Amount is null and Qty = 0 still there are 106 rows where order Status is not cancelld 

In [59]:
# Qty = 0, Amount is Null,not cancelled by Status, Courier Status = Cancelled
qty1[qty1['Courier Status']=="Cancelled"].shape
# Cancelled orders whose Status was never updated

(95, 21)

out of those 106, 95 order are cancelled by courier and Status was never updated<br>Updating this status 

In [60]:
# updating Status to Cancelled because courier status is also cancelled
before =(df['Status'] == "Cancelled").sum()
print("Number of rows Before updating Status",before)
row95 = (df['Qty'] == 0) & (df['Amount'].isna()) & (df['Status'] != "Cancelled") & (df['Courier Status'] == "Cancelled")
print("Number of rows Courier_status is cancelld but Status is not_cancelled",row95.sum()) 
df.loc[row95,'Status'] ="Cancelled"
after = (df['Status'] =='Cancelled').sum()
print(f"Before {before} \nAfter {after}")

Number of rows Before updating Status 18311
Number of rows Courier_status is cancelld but Status is not_cancelled 95
Before 18311 
After 18406


In [61]:
## Qty = 0, Amount is Null,not cancelled by Status, Courier Status = On the Way
qty2 = qty1[qty1['Courier Status']!="Cancelled"]
qty2.shape
# Contradictory records, no Qty and no Amount will drop it

(11, 21)

In [62]:
#drop 11 rows
df = df.drop(qty2.index)

In [63]:
df[['Date','Year','Year-Month','Month','Month Name']]

,Date,Year,Year-Month,Month,Month Name
0,2022-04-30,2022,2022-04,4,April
1,2022-04-30,2022,2022-04,4,April
2,2022-04-30,2022,2022-04,4,April
3,2022-04-30,2022,2022-04,4,April
4,2022-04-30,2022,2022-04,4,April
...,...,...,...,...,...
128971,2022-05-31,2022,2022-05,5,May
128972,2022-05-31,2022,2022-05,5,May
128973,2022-05-31,2022,2022-05,5,May
128974,2022-05-31,2022,2022-05,5,May


In [64]:
#month wise orders
march = df[df['Month']== 3]
april = df[df['Month']== 4]
may = df[df['Month']== 5]
june = df[df['Month']== 6]
print(f"March--{march.shape}\nApril--{april.shape}\nMay--{may.shape}\nJune{june.shape}")

March--(171, 21)
April--(48895, 21)
May--(42034, 21)
June(37697, 21)


Items with no location information

In [65]:
#missing values in ship-city, ship-state and ship-country
df['ship-city'] = df['ship-city'].fillna('Unknown')
df['ship-country'] = df['ship-country'].fillna('IN')
df['ship-postal-code'] = df['ship-postal-code'].fillna('000000')
df['ship-state'] = df['ship-state'].fillna('Unknown')

# 33 row has no location but has valid Amount, so marked as Unknown and postalcode is 000000

Fixing State Names

In [66]:
print(df['ship-state'].value_counts())
df['ship-state'].sort_values().unique()

ship-state
MAHARASHTRA      22233
KARNATAKA        17295
TAMIL NADU       11468
TELANGANA        11319
UTTAR PRADESH    10631
                 ...  
Nagaland             1
Mizoram              1
bihar                1
rajsthan             1
Rajsthan             1
Name: count, Length: 70, dtype: int64


array(['ANDAMAN & NICOBAR', 'ANDHRA PRADESH', 'APO', 'AR',
       'ARUNACHAL PRADESH', 'ASSAM', 'Arunachal Pradesh',
       'Arunachal pradesh', 'BIHAR', 'Bihar', 'CHANDIGARH',
       'CHHATTISGARH', 'Chandigarh', 'DADRA AND NAGAR', 'DELHI', 'Delhi',
       'GOA', 'Goa', 'Gujarat', 'HARYANA', 'HIMACHAL PRADESH',
       'JAMMU & KASHMIR', 'JHARKHAND', 'KARNATAKA', 'KERALA', 'LADAKH',
       'LAKSHADWEEP', 'MADHYA PRADESH', 'MAHARASHTRA', 'MANIPUR',
       'MEGHALAYA', 'MIZORAM', 'Manipur', 'Meghalaya', 'Mizoram',
       'NAGALAND', 'NL', 'Nagaland', 'New Delhi', 'ODISHA', 'Odisha',
       'Orissa', 'PB', 'PUDUCHERRY', 'PUNJAB', 'Pondicherry',
       'Puducherry', 'Punjab', 'Punjab/Mohali/Zirakpur', 'RAJASTHAN',
       'RJ', 'Rajasthan', 'Rajshthan', 'Rajsthan', 'SIKKIM', 'Sikkim',
       'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH',
       'UTTARAKHAND', 'Unknown', 'WEST BENGAL', 'bihar', 'delhi', 'goa',
       'orissa', 'punjab', 'rajasthan', 'rajsthan'], dtype=object)

In [67]:
#converting to title case,so that the state name with differnt case can marge
df['ship-state'] = df['ship-state'].str.title()
print(len(df['ship-state'].unique())) # we left with 48 from 70


48


In [68]:
stat = df.loc[(df['ship-state']=='Apo')|
              (df['ship-state']=='Ar')|
              (df['ship-state']=='NL')|
              (df['ship-state']=='Pb')|
              (df['ship-state']=='Rj')]
stat

,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Category,Size,Courier Status,Qty,...,ship-city,ship-state,ship-postal-code,ship-country,B2B,fulfilled-by,Year,Month,Month Name,Year-Month
25094,407-8361131-8501960,2022-04-15,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,Shirt,XL,Shipped,1,...,JAIPUR,Rj,302029.0,IN,True,Easy Ship,2022,4,April,2022-04
39783,404-6611202-9794717,2022-04-06,Shipped,Amazon,Amazon.in,Expedited,Shirt,3XL,Shipped,1,...,ZIRA,Pb,142044.0,IN,False,Not Easy Ship,2022,4,April,2022-04
45188,405-0034289-0259545,2022-04-03,Cancelled,Amazon,Amazon.in,Expedited,T-shirt,S,Cancelled,0,...,APO,Apo,959121.0,IN,False,Not Easy Ship,2022,4,April,2022-04
49145,405-8584178-5101966,2022-05-31,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,Shirt,XL,Shipped,1,...,JODHPUR,Rj,342009.0,IN,False,Easy Ship,2022,5,May,2022-05
51421,407-1375529-7104311,2022-05-30,Shipped,Amazon,Amazon.in,Expedited,Trousers,L,Shipped,1,...,ITANAGAR,Ar,791113.0,IN,False,Not Easy Ship,2022,5,May,2022-05


In [69]:
state_map = {
    'Rajshthan': 'Rajasthan',
    'Rajsthan': 'Rajasthan',
    'Rj': 'Rajasthan',
    'Orissa': 'Odisha',
    'Pondicherry': 'Puducherry',
    'New Delhi': 'Delhi',
    'Apo': 'Unknown',
    'Ar':'Arunachal Pradesh',
    'Nl':'Nagaland',
    'Pb':'Punjab',
    'Punjab/Mohali/Zirakpur':'Punjab'
}

In [70]:
df['ship-state'] = df['ship-state'].replace(state_map)

ship-state had 70 unique values due to case differences, spelling errors and old names. After standardizing, it has 37 (36 states and UTs plus Unknown)

In [71]:
print(df['ship-state'].sort_values().unique())
print(len(df['ship-state'].value_counts()))
print(df['ship-state'].value_counts())

['Andaman & Nicobar' 'Andhra Pradesh' 'Arunachal Pradesh' 'Assam' 'Bihar'
 'Chandigarh' 'Chhattisgarh' 'Dadra And Nagar' 'Delhi' 'Goa' 'Gujarat'
 'Haryana' 'Himachal Pradesh' 'Jammu & Kashmir' 'Jharkhand' 'Karnataka'
 'Kerala' 'Ladakh' 'Lakshadweep' 'Madhya Pradesh' 'Maharashtra' 'Manipur'
 'Meghalaya' 'Mizoram' 'Nagaland' 'Odisha' 'Puducherry' 'Punjab'
 'Rajasthan' 'Sikkim' 'Tamil Nadu' 'Telangana' 'Tripura' 'Unknown'
 'Uttar Pradesh' 'Uttarakhand' 'West Bengal']
37
ship-state
Maharashtra          22233
Karnataka            17295
Tamil Nadu           11468
Telangana            11319
Uttar Pradesh        10631
Delhi                 7035
Kerala                6573
West Bengal           5955
Andhra Pradesh        5419
Gujarat               4478
Haryana               4407
Rajasthan             2715
Madhya Pradesh        2529
Odisha                2138
Bihar                 2111
Punjab                1917
Assam                 1663
Uttarakhand           1551
Jharkhand             1455
Goa 

Fixing City 

In [72]:
len(df['ship-city'].unique())

8949

8949 unique cities in column

In [ ]:
#converting city names to title case
df['ship-city'] = df['ship-city'].str.title()
len(df['ship-city'].unique())

7292

8949 unique cities in the columns and after converting into title case we left with 7292 unnique city value

In [74]:
print(df['ship-city'].sort_values().value_counts().head(50))
print(df['ship-city'].sort_values().value_counts().head(20).sum())

ship-city
Bengaluru             11872
Hyderabad              9117
Mumbai                 7111
New Delhi              6329
Chennai                6280
Pune                   4616
Kolkata                2840
Gurugram               1951
Thane                  1875
Noida                  1626
Lucknow                1626
Ghaziabad              1484
Ahmedabad              1438
Navi Mumbai            1402
Bangalore              1360
Jaipur                 1061
Coimbatore              889
Visakhapatnam           858
Faridabad               816
Patna                   815
Bhubaneswar             780
Pimpri Chinchwad        753
Guwahati                740
Nagpur                  731
Vadodara                710
Thiruvananthapuram      679
Dehradun                632
Secunderabad            620
Greater Noida           564
Surat                   563
Indore                  557
Kalyan                  547
Kanpur                  541
Ernakulam               518
Bhopal                  502
Ranchi    

we can not check 7292 city values so checking top 50 city for old name new name and spalling mistakes.

In [75]:
# found one city with spelling mistake 
city_map = {
    'Bangalore': 'Bengaluru',
}
df['ship-city'] = df['ship-city'].replace(city_map)
len(df['ship-city'].unique())

7291

In [76]:
city_counts = df['ship-city'].value_counts()
total = len(df)

for n in [20, 50, 100]:
    rows = city_counts.head(n).sum()
    print(f"Top {n} cities: {rows} rows = {rows / total * 100:.1f}%")

Top 20 cities: 66146 rows = 51.4%
Top 50 cities: 80579 rows = 62.6%
Top 100 cities: 90111 rows = 70.0%


In [77]:
#creating new columns
df['is_cancelled'] = (df['Status']=="Cancelled")
df['amount_missing'] = (df['Amount'].isna())
df['amount_zero'] = (df['Amount']== 0 )

In [78]:
print(df['is_cancelled'].value_counts())
print(df['amount_missing'].value_counts())
print(df['amount_zero'].value_counts())

is_cancelled
False    110391
True      18406
Name: count, dtype: int64
amount_missing
False    121019
True       7778
Name: count, dtype: int64
amount_zero
False    126457
True       2340
Name: count, dtype: int64


In [ ]:
#fixing postal code type
df['ship-postal-code'] = df['ship-postal-code'].astype(str).str.replace(r'\.0$', '', regex=True)

In [80]:
# Checking the remaining columns and outliers
for col in ['Category','Size','Fulfilment','Courier Status','B2B','Sales Channel','ship-service-level']:
    print(df[col].value_counts(), '\n')

print(df['Qty'].value_counts().sort_index())
print(df.nlargest(10, 'Amount')[['Category','Size','Qty','Amount','Status']])


Category
T-shirt     50215
Shirt       49799
Blazzer     15488
Trousers    10606
Perfume      1157
Wallet        926
Socks         439
Shoes         164
Watch           3
Name: count, dtype: int64 

Size
M       22349
L       21774
XL      20517
XXL     17782
S       16752
3XL     14626
XS      11033
Free     2250
6XL       737
5XL       550
4XL       427
Name: count, dtype: int64 

Fulfilment
Amazon      89595
Merchant    39202
Name: count, dtype: int64 

Courier Status
Shipped       109341
On the Way      6852
Unshipped       6675
Cancelled       5929
Name: count, dtype: int64 

B2B
False    127926
True        871
Name: count, dtype: int64 

Sales Channel
Amazon.in     128673
Non-Amazon       124
Name: count, dtype: int64 

ship-service-level
Expedited    88513
Standard     40284
Name: count, dtype: int64 

Qty
0      12781
1     115629
2        340
3         32
4          9
5          2
8          1
9          1
13         1
15         1
Name: count, dtype: int64
       Category Siz

In [87]:
# Zero-Amount
zero = df[df['amount_zero']]
print(zero['Fulfilment'].value_counts())
print(df['Fulfilment'].value_counts())
print(zero['Category'].value_counts())
print(zero['Month Name'].value_counts())
print(df['amount_zero'].value_counts())

Fulfilment
Amazon      1527
Merchant     813
Name: count, dtype: int64
Fulfilment
Amazon      89595
Merchant    39202
Name: count, dtype: int64
Category
T-shirt     1012
Shirt        856
Blazzer      231
Trousers     174
Perfume       32
Wallet        22
Socks         12
Shoes          1
Name: count, dtype: int64
Month Name
April    830
June     762
May      748
Name: count, dtype: int64
amount_zero
False    126457
True       2340
Name: count, dtype: int64


2340 rows have Amount = 0 and none are cancelled.<br> They are kept and flagged with amount_zero, and are excluded only from average-price metrics

In [88]:
print(f"raw data--{raw_data.shape}\ndf clean data--{df.shape}")

raw data--(128976, 21)
df clean data--(128797, 24)
